# 🏆 SafeSense-VI: Vietnamese Hate Speech Detection## IT Got Talent 2025 - PhoBERT Training on Kaggle**Mô tả:** Notebook này huấn luyện mô hình PhoBERT để phân loại bình luận tiếng Việt thành 3 nhóm:- **0**: Bình thường (Clean)- **1**: Công kích/Xúc phạm (Offensive)  - **2**: Thù ghét (Hate Speech)**Yêu cầu Kaggle:**- GPU: T4 x2 hoặc P100- Internet: ON (để download model)- Dataset: Upload file `dataset_hate_speech_Vietnamese_KAGGLE.csv`

## 1️⃣ Cài đặt thư viện

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn seaborn

## 2️⃣ Import thư viện

In [ ]:
import pandas as pdimport numpy as npimport torchfrom torch.utils.data import Dataset, DataLoaderfrom transformers import (    AutoTokenizer,     AutoModelForSequenceClassification,    TrainingArguments,    Trainer,    EarlyStoppingCallback)from sklearn.model_selection import train_test_splitfrom sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_scoreimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings('ignore')# Kiểm tra GPUdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3️⃣ Cấu hình

In [ ]:
# ============== CẤU HÌNH ==============CONFIG = {    # Model    'model_name': 'vinai/phobert-base-v2',    'num_labels': 3,    'max_length': 256,        # Training    'batch_size': 16,    'learning_rate': 2e-5,    'num_epochs': 5,    'warmup_ratio': 0.1,    'weight_decay': 0.01,        # Data split    'test_size': 0.10,    'val_size': 0.10,    'random_state': 42,        # Paths    'data_path': '/kaggle/input/hate-speech-vietnamese/dataset_hate_speech_Vietnamese_KAGGLE.csv',    'output_dir': '/kaggle/working/phobert-hate-speech',}# Label mappingLABEL_NAMES = {    0: 'Clean (Bình thường)',    1: 'Offensive (Xúc phạm)',    2: 'Hate Speech (Thù ghét)'}print("Cấu hình:")for k, v in CONFIG.items():    print(f"  {k}: {v}")

## 4️⃣ Load và phân tích dữ liệu

In [ ]:
# Load datadf = pd.read_csv(CONFIG['data_path'])print(f"Tổng số mẫu: {len(df)}")print(f"\nCác cột: {list(df.columns)}")print(f"\nPhân bố label:")for label, count in df['label'].value_counts().sort_index().items():    pct = count / len(df) * 100    print(f"  {label} ({LABEL_NAMES[label]}): {count} ({pct:.1f}%)")# Hiển thị ví dụprint("\n" + "="*60)print("Ví dụ mỗi loại:")for label in [0, 1, 2]:    sample = df[df['label'] == label]['text'].iloc[0][:150]    print(f"\n[{label}] {LABEL_NAMES[label]}:")    print(f"  {sample}...")

In [ ]:
# Visualize phân bố labelfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Bar chartcolors = ['#2ecc71', '#f39c12', '#e74c3c']label_counts = df['label'].value_counts().sort_index()axes[0].bar([LABEL_NAMES[i].split(' ')[0] for i in label_counts.index],             label_counts.values, color=colors)axes[0].set_title('Phân bố Label', fontsize=14)axes[0].set_ylabel('Số lượng')for i, v in enumerate(label_counts.values):    axes[0].text(i, v + 50, str(v), ha='center', fontsize=12)# Pie chartaxes[1].pie(label_counts.values, labels=[LABEL_NAMES[i].split(' ')[0] for i in label_counts.index],            autopct='%1.1f%%', colors=colors, startangle=90)axes[1].set_title('Tỷ lệ phần trăm', fontsize=14)plt.tight_layout()plt.savefig('/kaggle/working/label_distribution.png', dpi=150)plt.show()

## 5️⃣ Chia tập Train / Validation / Test

In [ ]:
# Chia dữ liệu: Train (80%) / Val (10%) / Test (10%)# Stratified split để giữ tỷ lệ label# Bước 1: Tách Test settrain_val_df, test_df = train_test_split(    df,     test_size=CONFIG['test_size'],    stratify=df['label'],    random_state=CONFIG['random_state'])# Bước 2: Tách Validation set từ Trainval_ratio = CONFIG['val_size'] / (1 - CONFIG['test_size'])train_df, val_df = train_test_split(    train_val_df,    test_size=val_ratio,    stratify=train_val_df['label'],    random_state=CONFIG['random_state'])print("Kết quả chia dữ liệu:")print(f"  Train: {len(train_df)} mẫu ({len(train_df)/len(df)*100:.1f}%)")print(f"  Val:   {len(val_df)} mẫu ({len(val_df)/len(df)*100:.1f}%)")print(f"  Test:  {len(test_df)} mẫu ({len(test_df)/len(df)*100:.1f}%)")print("\nPhân bố label trong mỗi tập:")for name, data in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:    dist = data['label'].value_counts().sort_index()    print(f"  {name}: {dict(dist)}")

## 6️⃣ Tạo Dataset class

In [ ]:
class HateSpeechDataset(Dataset):    def __init__(self, texts, labels, tokenizer, max_length):        self.texts = texts        self.labels = labels        self.tokenizer = tokenizer        self.max_length = max_length        def __len__(self):        return len(self.texts)        def __getitem__(self, idx):        text = str(self.texts[idx])        label = self.labels[idx]                encoding = self.tokenizer(            text,            truncation=True,            padding='max_length',            max_length=self.max_length,            return_tensors='pt'        )                return {            'input_ids': encoding['input_ids'].flatten(),            'attention_mask': encoding['attention_mask'].flatten(),            'labels': torch.tensor(label, dtype=torch.long)        }

## 7️⃣ Load Tokenizer và Model

In [ ]:
print(f"Loading tokenizer: {CONFIG['model_name']}")tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])print(f"Loading model: {CONFIG['model_name']}")model = AutoModelForSequenceClassification.from_pretrained(    CONFIG['model_name'],    num_labels=CONFIG['num_labels'],    problem_type='single_label_classification')model.to(device)print(f"\nModel loaded on {device}")print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Tạo datasetstrain_dataset = HateSpeechDataset(    train_df['text'].values,    train_df['label'].values,    tokenizer,    CONFIG['max_length'])val_dataset = HateSpeechDataset(    val_df['text'].values,    val_df['label'].values,    tokenizer,    CONFIG['max_length'])test_dataset = HateSpeechDataset(    test_df['text'].values,    test_df['label'].values,    tokenizer,    CONFIG['max_length'])print(f"Train dataset: {len(train_dataset)} samples")print(f"Val dataset: {len(val_dataset)} samples")print(f"Test dataset: {len(test_dataset)} samples")

## 8️⃣ Định nghĩa Metrics

In [ ]:
def compute_metrics(eval_pred):    predictions, labels = eval_pred    predictions = np.argmax(predictions, axis=1)        accuracy = accuracy_score(labels, predictions)    f1_macro = f1_score(labels, predictions, average='macro')    f1_weighted = f1_score(labels, predictions, average='weighted')        return {        'accuracy': accuracy,        'f1_macro': f1_macro,        'f1_weighted': f1_weighted    }

## 9️⃣ Cấu hình Training

In [ ]:
training_args = TrainingArguments(    output_dir=CONFIG['output_dir'],        # Training params    num_train_epochs=CONFIG['num_epochs'],    per_device_train_batch_size=CONFIG['batch_size'],    per_device_eval_batch_size=CONFIG['batch_size'] * 2,        # Optimizer    learning_rate=CONFIG['learning_rate'],    weight_decay=CONFIG['weight_decay'],    warmup_ratio=CONFIG['warmup_ratio'],        # Evaluation    eval_strategy='epoch',    save_strategy='epoch',    load_best_model_at_end=True,    metric_for_best_model='f1_macro',    greater_is_better=True,        # Logging    logging_dir=f"{CONFIG['output_dir']}/logs",    logging_steps=50,    report_to='none',        # Others    seed=CONFIG['random_state'],    fp16=True,    dataloader_num_workers=2,    save_total_limit=2,)# Trainertrainer = Trainer(    model=model,    args=training_args,    train_dataset=train_dataset,    eval_dataset=val_dataset,    compute_metrics=compute_metrics,    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])print("Training configuration ready!")

## 🔟 Training

In [ ]:
print("="*60)print("🚀 BẮT ĐẦU TRAINING")print("="*60)train_result = trainer.train()print("\n" + "="*60)print("✅ TRAINING HOÀN TẤT")print("="*60)print(f"Training time: {train_result.metrics['train_runtime']:.1f}s")print(f"Training loss: {train_result.metrics['train_loss']:.4f}")

## 1️⃣1️⃣ Đánh giá trên Test Set

In [ ]:
print("="*60)print("📊 ĐÁNH GIÁ TRÊN TEST SET")print("="*60)test_results = trainer.evaluate(test_dataset)print(f"\nKết quả Test Set:")print(f"  Accuracy: {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)")print(f"  F1 Macro: {test_results['eval_f1_macro']:.4f}")print(f"  F1 Weighted: {test_results['eval_f1_weighted']:.4f}")

In [ ]:
# Predictions chi tiếtpredictions = trainer.predict(test_dataset)y_pred = np.argmax(predictions.predictions, axis=1)y_true = test_df['label'].values# Classification Reportprint("\n" + "="*60)print("CLASSIFICATION REPORT")print("="*60)target_names = ['Clean', 'Offensive', 'Hate Speech']print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

In [ ]:
# Confusion Matrixcm = confusion_matrix(y_true, y_pred)plt.figure(figsize=(10, 8))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',            xticklabels=target_names,            yticklabels=target_names)plt.title('Confusion Matrix - Test Set', fontsize=14)plt.xlabel('Predicted Label')plt.ylabel('True Label')plt.tight_layout()plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)plt.show()# Normalized confusion matrixcm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]plt.figure(figsize=(10, 8))sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',            xticklabels=target_names,            yticklabels=target_names)plt.title('Normalized Confusion Matrix - Test Set', fontsize=14)plt.xlabel('Predicted Label')plt.ylabel('True Label')plt.tight_layout()plt.savefig('/kaggle/working/confusion_matrix_normalized.png', dpi=150)plt.show()

## 1️⃣2️⃣ Phân tích lỗi

In [ ]:
# Tìm các mẫu dự đoán saitest_df_copy = test_df.copy()test_df_copy['predicted'] = y_predtest_df_copy['correct'] = test_df_copy['label'] == test_df_copy['predicted']wrong_predictions = test_df_copy[~test_df_copy['correct']]print(f"Số mẫu dự đoán sai: {len(wrong_predictions)} / {len(test_df)} ({len(wrong_predictions)/len(test_df)*100:.1f}%)")print("\n" + "="*60)print("MỘT SỐ MẪU DỰ ĐOÁN SAI:")print("="*60)for idx, row in wrong_predictions.head(10).iterrows():    print(f"\nText: {row['text'][:150]}...")    print(f"  True: {row['label']} ({target_names[row['label']]})")    print(f"  Pred: {row['predicted']} ({target_names[row['predicted']]})")

## 1️⃣3️⃣ Lưu Model

In [ ]:
# Lưu model tốt nhấtsave_path = '/kaggle/working/phobert-hate-speech-final'trainer.save_model(save_path)tokenizer.save_pretrained(save_path)print(f"Model đã lưu tại: {save_path}")# Lưu kết quảresults_summary = {    'model': CONFIG['model_name'],    'test_accuracy': test_results['eval_accuracy'],    'test_f1_macro': test_results['eval_f1_macro'],    'test_f1_weighted': test_results['eval_f1_weighted'],    'train_samples': len(train_df),    'val_samples': len(val_df),    'test_samples': len(test_df),}import jsonwith open('/kaggle/working/results_summary.json', 'w') as f:    json.dump(results_summary, f, indent=2)print("\nKết quả đã lưu vào results_summary.json")

## 1️⃣4️⃣ Demo Inference

In [ ]:
def predict_text(text, model, tokenizer, device):    model.eval()        encoding = tokenizer(        text,        truncation=True,        padding='max_length',        max_length=CONFIG['max_length'],        return_tensors='pt'    )        input_ids = encoding['input_ids'].to(device)    attention_mask = encoding['attention_mask'].to(device)        with torch.no_grad():        outputs = model(input_ids=input_ids, attention_mask=attention_mask)        probs = torch.softmax(outputs.logits, dim=1)        pred = torch.argmax(probs, dim=1).item()        return {        'label': pred,        'label_name': target_names[pred],        'confidence': probs[0][pred].item(),        'probabilities': {            target_names[i]: probs[0][i].item()             for i in range(3)        }    }# Demodemo_texts = [    "video này hay quá , cảm_ơn bạn đã chia_sẻ",    "thằng này ngu quá , không biết gì cả",    "bọn lgbt là đồ biến_thái , cần phải loại_bỏ"]print("="*60)print("DEMO INFERENCE")print("="*60)for text in demo_texts:    result = predict_text(text, model, tokenizer, device)    print(f"\nText: {text}")    print(f"  Prediction: {result['label_name']}")    print(f"  Confidence: {result['confidence']:.2%}")    print(f"  Probabilities: {result['probabilities']}")

## 1️⃣5️⃣ Tổng kết

In [ ]:
print("="*60)print("🏆 TỔNG KẾT - IT GOT TALENT 2025")print("="*60)print(f"\n📊 Model: {CONFIG['model_name']}")print(f"\n📈 Kết quả trên Test Set:")print(f"   • Accuracy: {test_results['eval_accuracy']*100:.2f}%")print(f"   • F1 Macro: {test_results['eval_f1_macro']:.4f}")print(f"   • F1 Weighted: {test_results['eval_f1_weighted']:.4f}")print(f"\n📁 Dữ liệu:")print(f"   • Train: {len(train_df)} mẫu")print(f"   • Validation: {len(val_df)} mẫu")print(f"   • Test: {len(test_df)} mẫu")print(f"\n💾 Model đã lưu tại: {save_path}")print("\n" + "="*60)print("✅ HOÀN TẤT!")print("="*60)